# 05 · Approach 3: Pretrained transformer (XLM-R / AfroXLMR / AfriBERTa) fine-tuning
Owner: **Member 4** · Runtime: Colab **GPU** (T4). Total for all experiments: about 1–1.5 h.

**Why this model.** Self-attention relates every token to every other token in one step, so
dependencies between distant parts of the article are modelled directly instead of being passed
along a recurrent chain (Vaswani et al., 2017). Pretraining gives the model knowledge of Swahili that
3,600 training articles cannot provide. We compare a general multilingual model, **XLM-R**
(Conneau et al., 2020), with **AfroXLMR**, the same model further adapted to 17 African languages
including Swahili (Alabi et al., 2022), and **AfriBERTa**, a smaller model pretrained only on African
languages (Ogueji et al., 2021). On MasakhaNEWS, a news-topic benchmark for 16 African languages
including Swahili, fine-tuned African-centric models of this kind are among the strongest baselines
(Adelani et al., 2023).
**Limitations:** 512 sub-word tokens is a hard limit, and articles are long; 100–280M parameters vs
~5M for the CNN; slow inference.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv` and `SampleSubmission.csv` (from Zindi) in a
Google Drive folder called `swahili_news`, and set `REPO_URL` below.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-news-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-news-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_news")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    for name in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        shutil.copy(DRIVE_DATA / name, REPO_DIR / "data" / name)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")             # e.g. downloaded word vectors persist
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")   # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import os
from functools import partial

import numpy as np
import torch

from src import config as C
from src.data import get_splits, split_frames, encode_labels, load_unlabelled
from src.datasets import TransformerDataset, transformer_collate
from src.models import HFTextClassifier
from src.pipeline import run_experiments, finalize
from src.text import GLUED, head_tail, tokenize, Vocab
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 4", "Transformer"
TINY = os.environ.get("TINY_TRANSFORMER") == "1"     # offline smoke test only: random 2-layer model
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
tr, va, te = split_frames(df)
ytr, yva, yte = (encode_labels(d.label) for d in (tr, va, te))
zindi = load_unlabelled()
K = len(C.CLASSES)

## Sub-word tokenisation with head / head+tail truncation
Only the glued-sentence repair is applied: the pretrained tokenisers expect natural, cased text.
Head+tail keeps the first and the last part of a long article. Sun et al. (2019) found that keeping
the head and the tail beats head-only truncation for long-document classification with BERT.

In [ ]:
class _TinyTokenizer:            # stands in for a Hugging Face tokenizer in offline smoke tests
    pad_token_id = 1

    def __init__(self):
        self.v = Vocab(tr.content.map(tokenize))

    def __call__(self, text, add_special_tokens=False):
        return {"input_ids": [i + 4 for i in self.v.encode(tokenize(text))]}

    def build_inputs_with_special_tokens(self, ids):
        return [0] + ids + [2]


_tok, _enc = {}, {}


def tokenizer(ckpt):
    if ckpt not in _tok:
        if TINY:
            _tok[ckpt] = _TinyTokenizer()
        else:
            from transformers import AutoTokenizer
            _tok[ckpt] = AutoTokenizer.from_pretrained(ckpt)
    return _tok[ckpt]


def encode(ckpt, head_frac=1.0):
    key = (ckpt, head_frac)
    if key not in _enc:
        tk = tokenizer(ckpt)
        body = C.MAX_TOKENS - 2
        def one(t):
            ids = tk(GLUED.sub(r"\1 ", t), add_special_tokens=False)["input_ids"]
            return tk.build_inputs_with_special_tokens(head_tail(ids, body, head_frac))
        _enc[key] = {n: {"input_ids": [one(t) for t in d.content]} for n, d in
                     {"tr": tr, "va": va, "te": te, "z": zindi}.items()}
        lens = np.array([len(tk(GLUED.sub(r"\1 ", t))["input_ids"]) for t in tr.content])
        print(f"{ckpt}: median {np.median(lens):.0f} sub-word tokens; {(lens > C.MAX_TOKENS).mean():.1%} of "
              f"training articles are longer than {C.MAX_TOKENS} and get truncated")
    return _enc[key]


def collate_for(ckpt):
    return partial(transformer_collate, pad_id=tokenizer(ckpt).pad_token_id)


def make_datasets(p):
    e = encode(p["checkpoint"], p.get("head_frac", 1.0))
    return TransformerDataset(e["tr"], ytr), TransformerDataset(e["va"], yva)


def build_model(p):
    return HFTextClassifier(p["checkpoint"], K, tiny_random=TINY,
                            vocab_size=len(tokenizer(p["checkpoint"]).v) + 4 if TINY else None)

## Experiments
Effective batch 32 (16 × 2 gradient accumulation), learning rate 3e-5 with warm-up (one-cycle),
early stopping on validation log loss.

In [ ]:
XLMR, AFROXLMR, AFRIBERTA = "FacebookAI/xlm-roberta-base", "Davlan/afro-xlmr-base", "castorini/afriberta_base"
HEAD_TAIL = 128 / 510       # first 128 + last 382 tokens (Sun et al., 2019)
DEFAULT_TRAIN = dict(epochs=5, lr=3e-5, batch_size=16, grad_accum=2, weight_decay=0.01, patience=2, num_workers=2)
EXPERIMENTS = [
    dict(id="TR-01", description="XLM-R base (100 languages), head 512 tokens",
         hypothesis="Multilingual pretraining plus self-attention beats models trained from scratch.",
         motivated_by="Starting point (compare with best CNN/RNN)", params={"checkpoint": XLMR}),
    dict(id="TR-02", description="AfroXLMR base (XLM-R adapted to African languages), head 512",
         hypothesis="Further pretraining on African-language text (incl. Swahili) improves Swahili news classification.",
         motivated_by="TR-01 + Alabi et al. (2022), Adelani et al. (2023)", params={"checkpoint": AFROXLMR}),
    dict(id="TR-03", description="AfroXLMR, head+tail truncation (128 + 382 tokens)",
         hypothesis="For articles longer than 512 tokens, the ending holds information the head misses.",
         motivated_by="EDA 4: many articles exceed 512 sub-words; EDA 8 keyword position; Sun et al. (2019)",
         params={"checkpoint": AFROXLMR, "head_frac": HEAD_TAIL}),
    dict(id="TR-04", description="AfroXLMR + class-weighted loss (best truncation)",
         hypothesis="Weighting helps the 11 burudani / 38 kimataifa training articles.",
         motivated_by="EDA 2 + per-class F1 of TR-02/03",
         params={"checkpoint": AFROXLMR, "head_frac": HEAD_TAIL}, train={"class_weights": True}),
    dict(id="TR-05", description="AfriBERTa base (pretrained only on African languages, smaller)",
         hypothesis="A smaller, African-only model is competitive at a fraction of the cost.",
         motivated_by="Cost/benefit question after TR-02", params={"checkpoint": AFRIBERTA}),
]
# TIP: after TR-03, set TR-04's head_frac to whichever truncation won.
if TINY:
    for e in EXPERIMENTS:
        e.setdefault("train", {}).update(batch_size=8, grad_accum=1, lr=1e-3, num_workers=0)

best, rows = None, []
for e in EXPERIMENTS:          # one call per experiment: each checkpoint has its own pad token
    b, s = run_experiments([e], build_model, make_datasets, member=MEMBER, model_name=MODEL, n_classes=K,
                           y_train=ytr, default_train=DEFAULT_TRAIN, collate_fn=collate_for(e["params"]["checkpoint"]))
    rows.append(s)
    if best is None or b["metrics"]["log_loss"] < best["metrics"]["log_loss"]:
        best = b
    else:
        del b
import pandas as pd
summary = pd.concat(rows, ignore_index=True)
print("Best on validation:", best["exp"]["id"])
summary.round(4)

> Fill the `observation` column in `experiments/log_transformer.csv`: XLM-R vs AfroXLMR vs AfriBERTa
> (does African-language pretraining matter?), head vs head+tail, class weights. Note the training
> time and parameter count too: that is the cost side of the trade-off.

## Final evaluation on the test split (once)

In [ ]:
p = best["exp"]["params"]
enc = encode(p["checkpoint"], p.get("head_frac", 1.0))
model, proba_test, metrics = finalize(best, build_model, TransformerDataset(enc["te"], yte), yte, te.id.values,
                                      C.CLASSES, key="transformer", display_name=f"Transformer ({p['checkpoint'].split('/')[-1]})",
                                      collate_fn=collate_for(p["checkpoint"]), save_weights=False,
                                      zindi_ds=TransformerDataset(enc["z"]), zindi_df=zindi)

In [ ]:
from src.utils import persist_outputs
persist_outputs()